# Applied Question #12

In [7]:
import numpy as np
import pandas as pd
from ISLP import load_data
from ISLP.bart import BART
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier

# -------------------------------
# 1. Load Data
# -------------------------------

carseats = load_data("Carseats")

# Create binary response
carseats["High"] = (carseats["Sales"] > 8).astype(int)

X = carseats.drop(columns=["Sales", "High"])
y = carseats["High"]

# Convert categorical variables
X = pd.get_dummies(X, drop_first=True)

# -------------------------------
# 2. Train/Test Split
# -------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# -------------------------------
# 3. Logistic Regression
# -------------------------------

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

log_model = LogisticRegression(max_iter=500)
log_model.fit(X_train_scaled, y_train)

log_acc = accuracy_score(y_test, log_model.predict(X_test_scaled))

# -------------------------------
# 4. Bagging
# -------------------------------

bag_model = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=500,
    random_state=42
)

bag_model.fit(X_train, y_train)
bag_acc = accuracy_score(y_test, bag_model.predict(X_test))

# -------------------------------
# 5. Random Forest
# -------------------------------

rf_model = RandomForestClassifier(
    n_estimators=500,
    max_features="sqrt",
    random_state=42
)

rf_model.fit(X_train, y_train)
rf_acc = accuracy_score(y_test, rf_model.predict(X_test))

# -------------------------------
# 6. Boosting
# -------------------------------

boost_model = GradientBoostingClassifier(
    n_estimators=1000,
    learning_rate=0.01,
    max_depth=3,
    random_state=42
)

boost_model.fit(X_train, y_train)
boost_acc = accuracy_score(y_test, boost_model.predict(X_test))

# -------------------------------
# 7. BART (ISLP Implementation)
# -------------------------------

# Convert to numeric float arrays
X_train_bart = X_train.astype(float).values
X_test_bart = X_test.astype(float).values
y_train_bart = y_train.astype(float).values

bart_model = BART(num_trees=200)
bart_model.fit(X_train_bart, y_train_bart)

bart_pred = bart_model.predict(X_test_bart)
bart_pred_binary = (bart_pred > 0.5).astype(int)

bart_acc = accuracy_score(y_test, bart_pred_binary)

# -------------------------------
# Final Comparison
# -------------------------------

print("\nModel Comparison")
print("-------------------------")
print("Logistic Regression:", round(log_acc, 3))
print("Bagging:", round(bag_acc, 3))
print("Random Forest:", round(rf_acc, 3))
print("Boosting:", round(boost_acc, 3))
print("BART:", round(bart_acc, 3))


Model Comparison
-------------------------
Logistic Regression: 0.758
Bagging: 0.742
Random Forest: 0.775
Boosting: 0.758
BART: 0.783
